In [ ]:
import pandas as pd
import geopandas as gpd
from dotenv import load_dotenv
import boto3
import json
import os

load_dotenv()

geocode_client = boto3.client('geo-places', region_name='us-west-2')

all_extracted = pd.read_csv('/Users/aayushkumbhare/Desktop/cafo-contamination/extracted.csv')
cleaned = pd.read_csv('/Users/aayushkumbhare/Desktop/cafo-contamination/extracted_logs.csv')

def get_address(df):
    for idx, row in df.iterrows():
        street = row['physical_street']
        city = row['physical_city']
        zipcode = row['physical_zip']
        state = 'CA'

        address = f"{street}, {city}, {state} {zipcode}"
        df.loc[idx, 'address'] = address
        #print(f"Added address for row {idx}")

    return df

all_extracted = get_address(all_extracted)

not_in_cleaned = all_extracted[~all_extracted['address'].isin(cleaned['address'])]




            operator_name      physical_street physical_city physical_county  \
0  Machado Dairy Farms #2  19405 E Mariposa RD      Stockton     San Joaquin   
1          Ed Nunes Dairy     17399 E Avena RD       Escalon     San Joaquin   
2                   Nunes     2618 E Reilly RD        Merced          Merced   
6                     NaN                  NaN           NaN             NaN   
8   Pedretti Ranches Inc.  2331 E Roosevelt RD       El Nido          Merced   

  physical_zip mailing_street mailing_city mailing_state  mailing_zip  \
0        95215  P.O. Box 4430      Manteca            CA      95337.0   
1        95320            NaN          NaN           NaN          NaN   
2        95340            NaN          NaN           NaN          NaN   
6        96002            NaN          NaN           NaN          NaN   
8        95317            NaN          NaN           NaN          NaN   

   operator_phone  ... total_land_applied_acres application_method  \
0  (209) 6

In [4]:
def get_coordinates(df):
    if 'latitude' not in df.columns:
        df['latitude'] = None

    if 'longitude' not in df.columns:
        df['longitude'] = None
    for idx, row in df.iterrows():
        address = row['address']

        try:
            response = geocode_client.geocode(
                QueryText=address,
                MaxResults=1
            )
        except Exception as e:
            print(f"Error adding coordinate, error: {e}")
        if response['ResultItems']:
            place = response['ResultItems'][0]
            coordinates = place['Position']
            print(f"Latitude: {coordinates[1]}; Longitude: {coordinates[0]}; address: {address}")


            df.at[idx, 'longitude'] = coordinates[0]
            df.at[idx, 'latitude'] = coordinates[1] 
        
            print(f"Added coordinates for address {address}")
        else:
            print(f"Coordinates for {address} not added")
    
    return df

dirty_df = get_coordinates(not_in_cleaned)

print(dirty_df.head())

dirty_df.to_csv('not_in_cleaned.csv')

Latitude: 37.87951; Longitude: -121.07751; address: 19405 E Mariposa RD, Stockton, CA 95215
Added coordinates for address 19405 E Mariposa RD, Stockton, CA 95215
Latitude: 37.8454; Longitude: -121.09484; address: 17399 E Avena RD, Escalon, CA 95320
Added coordinates for address 17399 E Avena RD, Escalon, CA 95320
Latitude: 37.25826; Longitude: -120.44054; address: 2618 E Reilly RD, Merced, CA 95340
Added coordinates for address 2618 E Reilly RD, Merced, CA 95340
Latitude: 40.49125; Longitude: -122.28651; address: nan, nan, CA 96002
Added coordinates for address nan, nan, CA 96002
Latitude: 37.14074; Longitude: -120.45143; address: 2331 E Roosevelt RD, El Nido, CA 95317
Added coordinates for address 2331 E Roosevelt RD, El Nido, CA 95317
Latitude: 38.32441; Longitude: -121.32512; address: 9950 Arno RD, Galt, CA 95632
Added coordinates for address 9950 Arno RD, Galt, CA 95632
Latitude: 37.40004; Longitude: -120.79874; address: 7581 N Merced AVE, Delhi, CA 95315
Added coordinates for addr